# Enhancing RAG with Contextual Embeddings

In [ ]:
%pip install python-dotenv

In [3]:
%%capture
%pip install --upgrade anthropic voyageai cohere elasticsearch pandas numpy

In [6]:
MODEL_NAME = "claude-haiku-4-5"

## Initialize Anthropic Client

We will start by initializing the Anthropic client that will be used to generate the contextual descriptions.

In [4]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv()

anthropic_client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))


In [9]:
response = anthropic_client.messages.create(
  model=MODEL_NAME,
  max_tokens=100,
  messages=[
    {"role": "user", "content": "Hello, Claude!"},
  ],
)

print(response.content)

[TextBlock(citations=None, text="Hello! It's nice to meet you. How can I help you today?", type='text')]


## Initialize Vector DB Class

This class will serve three main purposes:

1. Embedding Generation: Converts text chunks into vector embeddings using Voyage AI's embedding model.
2. Storage and Caching: Saved embeddings to disk to avoid recomputing them (saves time and money).
3. Similarity Search: Uses the most relevant chunks for a given query using cosine similarity.

For this guide, we will use a simple in-memory vector databse with pickle serialization. This makes the code easier to understand and requires no external dependencies.

In [10]:
import json
from multiprocessing import Value
import pickle
from typing import Any

import numpy as np
import voyageai
from tqdm import tqdm

class VectorDB:
    def __init__(self, name: str, api_key=None):
        if api_key is None:
            os.getenv("VOYAGE_API_KEY")
        self.client = voyageai.Client(api_key=api_key)
        self.name = name
        self.embeddings = []
        self.metadata = []
        self.query_cache = {}
        self.db_path = f"./data/{name}/vector_db.pkl"
    
    def load_data(self, dataset: list[dict[str, Any]]):
        if self.embeddings and self.metadata:
            print("Vector DB already loaded. Skipping data loading.")
            return
        if os.path.exists(self.db_path):
            print(f"Loading Vector DB from disk directory: {self.db_path}")
            self.load_db()
            return
        
        texts_to_embed = []
        metadata = []
        total_chunks = sum(len(doc["chunks"]) for doc in dataset)

        with tqdm(total=total_chunks, desc="Processing chunks") as pbar:
            for doc in dataset:
                for chunk in doc["chunks"]:
                    texts_to_embed.append(chunk["content"])
                    metadata.append({
                        "doc_id": doc["doc_id"],
                        "original_uuid": doc["original_uuid"],
                        "chunk_id": chunk["chunk_id"],
                        "original_index": chunk["original_index"],
                        "content": chunk["content"],
                    }
                )
                pbar.update(1)
        
        self._embed_and_store(texts_to_embed, metadata)
        self.save_db()

        print(f"Vector database loaded and saved. Total chunks processed: {len(texts_to_embed)}")

    def _embed_and_store(self, texts:list[str], data:list[dict[str, Any]]):
        batch_size = 128
        with tqdm(total=len(texts), desc="Embedding chunks") as pbar:
            result = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i+batch_size]
                batch_result = self.client.embed(batch, model='voyage-2').embeddings
                result.extend(batch_result)
                pbar.update(len(batch))

        self.embeddings = result
        self.metadata = data
    

    def search(self, query: str, k: int = 20) -> list[dict[str, Any]]:
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.client.embed([query], model="voyage-2").embeddings[0]
            self.query_cache[query] = query_embedding

        if not self.embeddings:
            raise ValueError("No data loaded in the vector databse")
        
        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[::-1][:k]

        top_results = []
        for idx in top_indices:
            result = {
                "metadata": self.metadata[idx],
                "similarity": float(similarities[idx])
            }
            top_results.append(result)
        
        return top_results

    def save_db(self):
        data = {
            "embeddings": self.embeddings,
            "metadata": self.metadata,
            "query_cache": self.query_cache
        }
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        with open(self.db_path, "wb") as file:
            pickle.dump(data, file)
    
    def load_db(self):
        if not os.path.exists(self.db_path):
            raise ValueError("Vector database file not found. Use load_data to create a new database")
        
        with open(self.db_path, "rb") as file:
            data = pickle.load(file)
        self.embeddings = data["embeddings"]
        self.metadata = data["metadata"]
        self.query_cache = data["query_cache"]
        


        

/Users/josepujol/Documents/AI_ML/Papers/ML-Papers/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
